# Notebook 2: Baselines

Runs four baselines and saves all predictions to `predictions/`.

| Baseline | Model | Retrieval |
|----------|-------|-----------|
| B1: BERT extractive | RoBERTa-SQuAD | None |
| B2: BM25 + BERT extractive | RoBERTa-SQuAD | BM25 top-5 |
| B3: LLM zero-shot | Qwen2.5-14B-Instruct | None |
| B4: LLM few-shot | Qwen2.5-14B-Instruct | None |

**Requires:** `corpus.json`, `titles.json`, `bm25_index.pkl` from notebook 01.

In [ ]:
# Google Drive
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/FRAMES'

import os
os.makedirs(os.path.join(DATA_DIR, 'predictions'), exist_ok=True)

Mounted at /content/drive


In [ ]:
!pip install transformers datasets rank_bm25 torch tqdm accelerate -q

In [ ]:
import json, pickle, os, re, ast, collections
from tqdm.auto import tqdm
import torch
from datasets import load_dataset

# Load FRAMES
frames = load_dataset('google/frames-benchmark')['test']
print(f'FRAMES: {len(frames)} questions')

# Load BM25 index and corpus
with open(os.path.join(DATA_DIR, 'corpus.json'), encoding='utf-8') as f:
    corpus = json.load(f)
with open(os.path.join(DATA_DIR, 'titles.json'), encoding='utf-8') as f:
    titles = json.load(f)
with open(os.path.join(DATA_DIR, 'bm25_index.pkl'), 'rb') as f:
    bm25 = pickle.load(f)
print(f'Corpus: {len(corpus)} articles, BM25 index loaded')

def get_wiki_links(row):
    links = row['wiki_links']
    if isinstance(links, str):
        try:
            links = ast.literal_eval(links)
        except:
            links = [links]
    return [l for l in links if isinstance(l, str) and len(l) > 3]

# Only evaluating 50% of the dataset to save runtime
EVAL_SUBSET = 0.5


if EVAL_SUBSET < 1.0:
    import random
    from collections import defaultdict
    random.seed(42)
    type_indices = defaultdict(list)
    for i, row in enumerate(frames):
        key = tuple(sorted(row.get('reasoning_types') or ['Unknown']))
        type_indices[key].append(i)
    subset_indices = set()
    for key, indices in type_indices.items():
        k = max(1, round(len(indices) * EVAL_SUBSET))
        subset_indices.update(random.sample(indices, k))
    frames = [frames[i] for i in sorted(subset_indices)]
    print(f'Subset: {len(frames)} questions ({EVAL_SUBSET:.0%} stratified, seed=42)')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

test.tsv: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/824 [00:00<?, ? examples/s]

FRAMES: 824 questions
Corpus: 3963 articles, BM25 index loaded
Subset: 412 questions (50% stratified, seed=42)


In [ ]:
def retrieve_bm25(query, k=5):
    tokens = query.lower().split()
    scores = bm25.get_scores(tokens)
    top_idx = scores.argsort()[-k:][::-1]
    return [(titles[i], corpus[titles[i]]) for i in top_idx]

def get_top_passages(content, query, n=1, passage_chars=500, stride=250):
    passages = []
    for start in range(0, len(content), stride):
        chunk = content[start:start + passage_chars]
        if len(chunk) < 50:
            continue
        passages.append(chunk)
    if not passages:
        return content[:passage_chars]
    query_words = set(query.lower().split())
    scored = sorted(passages, key=lambda p: -len(query_words & set(p.lower().split())))
    return ' '.join(scored[:n])

def save_predictions(name, preds):
    path = os.path.join(DATA_DIR, 'predictions', f'{name}.json')
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(preds, f, ensure_ascii=False, indent=2)
    print(f'Saved {len(preds)} predictions -> {path}')

In [ ]:
from transformers import pipeline as hf_pipeline

qa_pipe = hf_pipeline(
    'question-answering',
    model='deepset/roberta-base-squad2',
    device=0 if torch.cuda.is_available() else -1
)

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

RoBERTa-SQuAD loaded


In [ ]:
# B1: BERT extractive, NO retrieval
# RoBERTa-SQuAD can only extract spans from a context, it has no generative memory.
# We pass the question itself as the context so the model at least tries to find an answer
# within the question text. We expect it will almost always be wrong
# This will show the floor: extractive models without retrieved context can't answer FRAMES questions.

b1_preds = []
for row in tqdm(frames, desc='B1 BERT no-retrieval'):
    out = qa_pipe(question=row['Prompt'], context=row['Prompt'])
    b1_preds.append({
        'id': row.get('id', ''),
        'question': row['Prompt'],
        'gold_answer': row['Answer'],
        'predicted': out['answer'],
        'score': out['score'],
        'reasoning_types': row['reasoning_types'],
        'n_gold_articles': len(get_wiki_links(row)),
        'gold_wiki_links': get_wiki_links(row),
        'retrieved_titles': []
    })

save_predictions('b1_bert_no_retrieval', b1_preds)

B1 BERT no-retrieval:   0%|          | 0/412 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Saved 412 predictions -> /content/drive/MyDrive/FRAMES/predictions/b1_bert_no_retrieval.json


In [ ]:
# B2: BM25 top-5 -> best passage per article -> RoBERTa extractive

b2_preds = []
for row in tqdm(frames, desc='B2 BM25+BERT'):
    retrieved = retrieve_bm25(row['Prompt'], k=5)
    context = ' '.join([get_top_passages(content, row['Prompt'], n=1, passage_chars=500)
                        for _, content in retrieved])
    context = context[:3000]

    out = qa_pipe(question=row['Prompt'], context=context)
    b2_preds.append({
        'id': row.get('id', ''),
        'question': row['Prompt'],
        'gold_answer': row['Answer'],
        'predicted': out['answer'],
        'score': out['score'],
        'reasoning_types': row['reasoning_types'],
        'n_gold_articles': len(get_wiki_links(row)),
        'gold_wiki_links': get_wiki_links(row),
        'retrieved_titles': [t for t, _ in retrieved]
    })

save_predictions('b2_bm25_bert', b2_preds)

B2 BM25+BERT:   0%|          | 0/412 [00:00<?, ?it/s]

Saved 412 predictions -> /content/drive/MyDrive/FRAMES/predictions/b2_bm25_bert.json


## Baselines 3 & 4: Generative LLM

Loads Qwen2.5-7B-Instruct. B3 = zero-shot (question only). B4 = few-shot (3 FRAMES examples prepended).

**GPU memory:** ~14GB in float16. Fits on Colab T4. If you run out of memory, switch to `load_in_4bit=True` below.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = 'Qwen/Qwen2.5-14B-Instruct'

print(f'Loading {MODEL_ID}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
)
model.eval()
print('Model loaded')
print(f'Device: {next(model.parameters()).device}')

Loading Qwen/Qwen2.5-14B-Instruct...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded
Device: cuda:0


In [ ]:
def llm_generate(prompt, max_new_tokens=256):
    """Generate a response given a plain text prompt."""
    messages = [{'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id
        )
    new_tokens = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# Just a test
test = llm_generate('What is the capital of Norway?')
print('Test output:', test)

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Test output: The capital of Norway is Oslo.


In [ ]:
# B3: LLM zero-shot
# Prompt made by ChatGPT
ZERO_SHOT_TEMPLATE = """Answer the following question. Be concise — give only the answer, not a full explanation.

Question: {question}
Answer:"""

b3_preds = []
for row in tqdm(frames, desc='B3 LLM zero-shot'):
    prompt = ZERO_SHOT_TEMPLATE.format(question=row['Prompt'])
    answer = llm_generate(prompt, max_new_tokens=128)
    b3_preds.append({
        'id': row.get('id', ''),
        'question': row['Prompt'],
        'gold_answer': row['Answer'],
        'predicted': answer,
        'reasoning_types': row['reasoning_types'],
        'n_gold_articles': len(get_wiki_links(row)),
        'gold_wiki_links': get_wiki_links(row),
        'retrieved_titles': []
    })

save_predictions('b3_llm_zeroshot', b3_preds)

B3 LLM zero-shot:   0%|          | 0/412 [00:00<?, ?it/s]

Saved 412 predictions -> /content/drive/MyDrive/FRAMES/predictions/b3_llm_zeroshot.json


In [ ]:
# B4: LLM few-shot
# Prompt made by ChatGPT

FEW_SHOT_EXAMPLES = [
    {
        'question': 'What is the capital city of the country that hosts the FIFA World Cup most recently before 2022?',
        'answer': 'Moscow'
    },
    {
        'question': 'How many Academy Awards has the director of Inception won in total?',
        'answer': '0'
    },
    {
        'question': 'What year was the oldest university in the United States founded?',
        'answer': '1636'
    },
]

def build_few_shot_prompt(question):
    examples = '\n'.join(
        f'Question: {ex["question"]}\nAnswer: {ex["answer"]}'
        for ex in FEW_SHOT_EXAMPLES
    )
    return f"""Answer questions concisely based on your knowledge. Here are some examples:

{examples}

Now answer:
Question: {question}
Answer:"""

b4_preds = []
for row in tqdm(frames, desc='B4 LLM few-shot'):
    prompt = build_few_shot_prompt(row['Prompt'])
    answer = llm_generate(prompt, max_new_tokens=128)
    b4_preds.append({
        'id': row.get('id', ''),
        'question': row['Prompt'],
        'gold_answer': row['Answer'],
        'predicted': answer,
        'reasoning_types': row['reasoning_types'],
        'n_gold_articles': len(get_wiki_links(row)),
        'gold_wiki_links': get_wiki_links(row),
        'retrieved_titles': []
    })

save_predictions('b4_llm_fewshot', b4_preds)

B4 LLM few-shot:   0%|          | 0/412 [00:00<?, ?it/s]

Saved 412 predictions -> /content/drive/MyDrive/FRAMES/predictions/b4_llm_fewshot.json


In [ ]:
# F1  EM check
import collections

def normalize(text):
    text = text.lower().strip()
    text = re.sub(r'[^\w\s]', '', text)
    return text

def token_f1(pred, gold):
    pred_tokens = normalize(pred).split()
    gold_tokens = normalize(gold).split()
    common = collections.Counter(pred_tokens) & collections.Counter(gold_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    p = num_same / len(pred_tokens)
    r = num_same / len(gold_tokens)
    return 2 * p * r / (p + r)

def exact_match(pred, gold):
    return float(normalize(pred) == normalize(gold))

def evaluate_predictions(preds, label):
    f1s = [token_f1(p['predicted'], p['gold_answer']) for p in preds]
    ems = [exact_match(p['predicted'], p['gold_answer']) for p in preds]
    print(f'{label:30s}  F1={sum(f1s)/len(f1s):.3f}  EM={sum(ems)/len(ems):.3f}')

print('F1 / EM:')
evaluate_predictions(b1_preds, 'B1: BERT no-retrieval')
evaluate_predictions(b2_preds, 'B2: BM25 + BERT')
evaluate_predictions(b3_preds, 'B3: LLM zero-shot')
evaluate_predictions(b4_preds, 'B4: LLM few-shot')

--- Token F1 / EM (development proxy) ---
B1: BERT no-retrieval           F1=0.063  EM=0.000
B2: BM25 + BERT                 F1=0.065  EM=0.032
B3: LLM zero-shot               F1=0.145  EM=0.044
B4: LLM few-shot                F1=0.080  EM=0.002

Note: LLM-as-judge (notebook 04) will give higher and more accurate scores.
